Variable Notes

pclass: A proxy for socio-economic status (SES)

1st = Upper

2nd = Middle

3rd = Lower

age: Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5

sibsp: The dataset defines family relations in this way...

Sibling = brother, sister, stepbrother, stepsister

Spouse = husband, wife (mistresses and fiancés were ignored)

parch: The dataset defines family relations in this way...

Parent = mother, father

Child = daughter, son, stepdaughter, stepson

Some children travelled only with a nanny, therefore parch=0 for them.

In [1]:
url = r"C:\Users\princ\OneDrive\Documents\Chapter 1\Python\Machine Learning\ML Data\Titanic Data.csv"

Importing The Dependencies

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score

Data Collection and Preprocessing

In [3]:
#Converting the dataset to a pandas dataframe
data = pd.read_csv(url)

In [4]:
#Printing first five rows of the data
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Survived:

0-- No

1-- Yes

In [5]:
data.shape

(891, 12)

In [6]:
#Checking for any missing value
data.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [7]:
#REPLACING ALL THE MISSING  VALUES OF THE AGE COLUMN WITH ITS MEAN
I = data['Age'].mean()
data['Age'].fillna(I,inplace = True)
data.isnull().sum()

C:\Users\princ\AppData\Local\Temp\ipykernel_33100\1523154485.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Age'].fillna(I,inplace = True)


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [8]:
#Dropping the Cabin Column
data = data.drop(columns = 'Cabin',axis = 1)

In [9]:
#Dropping all the  missing values of the Embarked column
data = data.dropna(subset = ['Embarked'])
data.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

In [10]:
print(data['Age'].mean())

29.65344637067425


In [11]:
data['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [12]:
data['Parch'].value_counts()

Parch
0    676
1    118
2     80
5      5
3      5
4      4
6      1
Name: count, dtype: int64

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['Name'] = le.fit_transform(data['Name'])
data['Sex'] = le.fit_transform(data['Sex'])
data['Ticket'] = le.fit_transform(data['Ticket'])
data['Embarked'] = le.fit_transform(data['Embarked'])

In [14]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,108,1,22.0,1,0,522,7.2500,2
1,2,1,1,190,0,38.0,1,0,595,71.2833,0
2,3,1,3,353,0,26.0,0,0,668,7.9250,2
3,4,1,1,272,0,35.0,1,0,48,53.1000,2
4,5,0,3,15,1,35.0,0,0,471,8.0500,2


In [15]:
#NOW CHECKING THE CORRELATION BETWEEN THE DATA
data.corr()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
PassengerId,1.000000,-0.005028,-0.035330,-0.041324,0.043136,0.030300,-0.057686,-0.001657,-0.056852,0.012703,0.013166
Survived,-0.005028,1.000000,-0.335549,-0.059075,-0.541585,-0.074673,-0.034040,0.083151,-0.160931,0.255290,-0.169718
Pclass,-0.035330,-0.335549,1.000000,0.054837,0.127741,-0.327954,0.081656,0.016824,0.315959,-0.548193,0.164681
Name,-0.041324,-0.059075,0.054837,1.000000,0.022087,0.054221,-0.016558,-0.048533,0.049406,-0.050396,-0.006849
Sex,0.043136,-0.541585,0.127741,0.022087,1.000000,0.089434,-0.116348,-0.247508,0.055024,-0.179958,0.110320
Age,0.030300,-0.074673,-0.327954,0.054221,0.089434,1.000000,-0.231875,-0.178232,-0.063799,0.088604,-0.028927
SibSp,-0.057686,-0.034040,0.081656,-0.016558,-0.116348,-0.231875,1.000000,0.414542,0.077995,0.160887,0.068900
Parch,-0.001657,0.083151,0.016824,-0.048533,-0.247508,-0.178232,0.414542,1.000000,0.018409,0.217532,0.040449
Ticket,-0.056852,-0.160931,0.315959,0.049406,0.055024,-0.063799,0.077995,0.018409,1.000000,-0.010562,0.000271
Fare,0.012703,0.255290,-0.548193,-0.050396,-0.179958,0.088604,0.160887,0.217532,-0.010562,1.000000,-0.226311


In [16]:
#Seprating the data to target and features
X = data.drop(columns = 'Survived',axis = 1)
Y = data['Survived']

Importing the Models

In [17]:
#importing the models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [18]:
#list of models
models = [LogisticRegression(max_iter = 10000),KNeighborsClassifier(),RandomForestClassifier(random_state = 2)]

In [19]:
def compare_model_cross_validation():
    for model in models:
        
        cv_score = cross_val_score(model,X,Y,cv = 5)
        mean_accuracy = sum(cv_score)/len(cv_score)
        mean_accuracy = mean_accuracy*100
        mean_accuracy = round(mean_accuracy,2)
        
        print('Cross Validation accuracy for the ',model,'=',cv_score)
        print('Accuracy score of the ',model,'=',mean_accuracy,'%')
        print('-----------------------------------------------------')

In [20]:
compare_model_cross_validation()

Cross Validation accuracy for the  LogisticRegression(max_iter=10000) = [0.7752809  0.78651685 0.78651685 0.7752809  0.81920904]
Accuracy score of the  LogisticRegression(max_iter=10000) = 78.86 %
-----------------------------------------------------
Cross Validation accuracy for the  KNeighborsClassifier() = [0.57865169 0.59550562 0.56741573 0.61797753 0.6440678 ]
Accuracy score of the  KNeighborsClassifier() = 60.07 %
-----------------------------------------------------
Cross Validation accuracy for the  RandomForestClassifier(random_state=2) = [0.80337079 0.8258427  0.84831461 0.84831461 0.84745763]
Accuracy score of the  RandomForestClassifier(random_state=2) = 83.47 %
-----------------------------------------------------


As we can see the best model between the all of them is RandomForestClassifier with the Accuracy of 83.47% 